In [ ]:
!pip install kagglehub pandas

In [11]:
import sys
import os

# 0. Agregar la ruta principal del proyecto para que Python encuentre 'src'
sys.path.append(os.path.abspath('..'))

from src.inf8239_u01.data import download_csv
import pandas as pd

# 1. Ejecutar la descarga reproducible encapsulada
KAGGLE_SOURCE = "ahmadrafiee/bank-personal-loan"
path = download_csv(KAGGLE_SOURCE)

# 2. Cargar el dataframe desde la ruta estructurada del proyecto
df = pd.read_csv(path)
display(df.head())

Descargando dataset desde la API de Kaggle: ahmadrafiee/bank-personal-loan...
Dataset guardado reproduciblemente en: data\raw\dataset.csv


,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,1,25,1,49,91107,4,1/60,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1/50,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1/00,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2/70,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1/00,2,0,0,0,0,0,1


In [4]:
# 1. Corregir la columna CCAvg: reemplazar '/' por '.' y convertir a número flotante
df['CCAvg'] = df['CCAvg'].astype(str).str.replace('/', '.').astype(float)

# 2. Eliminar columnas innecesarias para la predicción
df = df.drop(['ID', 'ZIP Code'], axis=1)

# 3. Verificar los tipos de datos actualizados y buscar valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 5000 non-null   int64  
 1   Experience          5000 non-null   int64  
 2   Income              5000 non-null   int64  
 3   Family              5000 non-null   int64  
 4   CCAvg               5000 non-null   float64
 5   Education           5000 non-null   int64  
 6   Mortgage            5000 non-null   int64  
 7   Personal Loan       5000 non-null   int64  
 8   Securities Account  5000 non-null   int64  
 9   CD Account          5000 non-null   int64  
 10  Online              5000 non-null   int64  
 11  CreditCard          5000 non-null   int64  
dtypes: float64(1), int64(11)
memory usage: 468.9 KB


In [5]:
from sklearn.model_selection import train_test_split

# 1. Separar características (X) del target (y)
X = df.drop('Personal Loan', axis=1)
y = df['Personal Loan']

# 2. Dividir en entrenamiento (80%) y prueba (20%)
# Usamos stratify=y para mantener la proporción de préstamos aceptados en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Set de Entrenamiento: {X_train.shape[0]} registros")
print(f"Set de Prueba: {X_test.shape[0]} registros")

Set de Entrenamiento: 4000 registros
Set de Prueba: 1000 registros


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

# 1. Escalar los datos (Ajustar SOLO en entrenamiento para evitar fuga de información)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Entrenar el modelo base (Adivinará la clase mayoritaria: "0 - No acepta préstamo")
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_scaled, y_train)
dummy_pred = dummy.predict(X_test_scaled)

print("--- Reporte Línea Base (Dummy Classifier) ---")
print(classification_report(y_test, dummy_pred, zero_division=0))

--- Reporte Línea Base (Dummy Classifier) ---
              precision    recall  f1-score   support

           0       0.90      1.00      0.95       904
           1       0.00      0.00      0.00        96

    accuracy                           0.90      1000
   macro avg       0.45      0.50      0.47      1000
weighted avg       0.82      0.90      0.86      1000



In [7]:
from sklearn.svm import SVC

# 1. Instanciar el modelo SVM (kernel RBF por defecto, ideal para relaciones no lineales)
svm_model = SVC(kernel='rbf', class_weight='balanced', random_state=42)

# 2. Entrenar el modelo con los datos escalados
svm_model.fit(X_train_scaled, y_train)

# 3. Generar las predicciones
svm_pred = svm_model.predict(X_test_scaled)

# 4. Evaluar el rendimiento
print("--- Reporte Modelo SVM (Pesos Balanceados) ---")
print(classification_report(y_test, svm_pred))

--- Reporte Modelo SVM (Pesos Balanceados) ---
              precision    recall  f1-score   support

           0       1.00      0.95      0.97       904
           1       0.69      0.97      0.81        96

    accuracy                           0.95      1000
   macro avg       0.84      0.96      0.89      1000
weighted avg       0.97      0.95      0.96      1000



In [8]:
from sklearn.model_selection import GridSearchCV

# 1. Definir la malla de hiperparámetros a explorar
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01]
}

# 2. Configurar la validación cruzada (optimizando por f1_macro para proteger a la clase minoritaria)
grid_search = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro', 
    n_jobs=-1 # Usa todos los núcleos del procesador para mayor velocidad
)

# 3. Entrenar la malla con los datos de entrenamiento
grid_search.fit(X_train_scaled, y_train)

print("Mejores hiperparámetros encontrados:", grid_search.best_params_)

# 4. Evaluar el mejor modelo encontrado contra los datos de prueba
best_svm = grid_search.best_estimator_
best_pred = best_svm.predict(X_test_scaled)

print("\n--- Reporte Modelo SVM (Optimizado con GridSearchCV) ---")
print(classification_report(y_test, best_pred))

Mejores hiperparámetros encontrados: {'C': 100, 'gamma': 0.1}

--- Reporte Modelo SVM (Optimizado con GridSearchCV) ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       904
           1       0.87      0.88      0.87        96

    accuracy                           0.97      1000
   macro avg       0.93      0.93      0.93      1000
weighted avg       0.98      0.97      0.98      1000



In [9]:
import joblib
import os

# Crear el directorio si no existe
os.makedirs('models', exist_ok=True)

# Serializar el modelo optimizado y el escalador de datos
joblib.dump(best_svm, 'models/svm_bank_loan_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

print("Modelo optimizado y escalador guardados con éxito en la carpeta 'models'.")

Modelo optimizado y escalador guardados con éxito en la carpeta 'models'.


In [12]:
import pandas as pd

# Cargar el dataset desde la ruta estructurada
df = pd.read_csv("data/raw/dataset.csv")

# Imprimir métricas de reconocimiento
print("Dimensiones:", df.shape)
print("\nTipos de datos:\n", df.dtypes)
print("\nHead:\n")
display(df.head())
print("\nTail:\n")
display(df.tail())

# Validación de control
assert not df.empty, "El dataset está vacío"

Dimensiones: (5000, 14)

Tipos de datos:
 ID                    int64
Age                   int64
Experience            int64
Income                int64
ZIP Code              int64
Family                int64
CCAvg                   str
Education             int64
Mortgage              int64
Personal Loan         int64
Securities Account    int64
CD Account            int64
Online                int64
CreditCard            int64
dtype: object

Head:



,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,1,25,1,49,91107,4,1/60,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1/50,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1/00,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2/70,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1/00,2,0,0,0,0,0,1



Tail:



,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
4995,4996,29,3,40,92697,1,1/90,3,0,0,0,0,1,0
4996,4997,30,4,15,92037,4,0/40,1,85,0,0,0,1,0
4997,4998,63,39,24,93023,2,0/30,3,0,0,0,0,0,0
4998,4999,65,40,49,90034,3,0/50,2,0,0,0,0,1,0
4999,5000,28,4,83,92612,3,0/80,1,0,0,0,0,1,1


In [13]:
# Construir el dataframe de auditoría
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

# Validar registros duplicados
print("Duplicados exactos en el dataset:", df.duplicated().sum())
display(audit)

Duplicados exactos en el dataset: 0


,tipo,ausentes,porcentaje_ausente,unicos
ID,int64,0,0.0,5000
Age,int64,0,0.0,45
Experience,int64,0,0.0,47
Income,int64,0,0.0,162
ZIP Code,int64,0,0.0,467
Family,int64,0,0.0,4
CCAvg,str,0,0.0,108
Education,int64,0,0.0,3
Mortgage,int64,0,0.0,347
Personal Loan,int64,0,0.0,2


In [14]:
TARGET = "Personal Loan"
DROP_COLUMNS = ["ID", "ZIP Code"] 

assert TARGET in df.columns
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print("Distribución de la variable Target:")
print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2

Distribución de la variable Target:
Personal Loan
0    4520
1     480
Name: count, dtype: int64


In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- INTERVENCIÓN CRÍTICA DE LIMPIEZA ---
# Corregir CCAvg para que el pipeline la reconozca como numérica
X['CCAvg'] = X['CCAvg'].astype(str).str.replace('/', '.').astype(float)
# ----------------------------------------

# 1. Separar tipos de datos
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

# 2. Crear pipelines específicos
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 3. Ensamblar el preprocesador
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print(f"Numéricas ({len(num_cols)}):", num_cols)
print(f"Categóricas ({len(cat_cols)}):", cat_cols)

Numéricas (11): ['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Education', 'Mortgage', 'Securities Account', 'CD Account', 'Online', 'CreditCard']
Categóricas (0): []


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
dummy=Pipeline([("prep",preprocess),("model",DummyClassifier(strategy="most_frequent"))])
svm=Pipeline([("prep",preprocess),("model",SVC(C=1,gamma="scale",probability=True,random_state=42))])
for name,model in {"dummy":dummy,"svm":svm}.items():
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    print(name, f1_score(yte,pred,average="macro"))
print(classification_report(yte,svm.predict(Xte)))

dummy 0.47478991596638653


c:\Users\lenne\OneDrive\Desktop\Maestria Ciencia de Datos e Inteligencia Artificial\Ciencia de Datos II\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.9307575128098602
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       904
           1       0.97      0.79      0.87        96

    accuracy                           0.98      1000
   macro avg       0.98      0.89      0.93      1000
weighted avg       0.98      0.98      0.98      1000

